In [ ]:
timeout = dbutils.widgets.get("timeout")

In [ ]:
import json
import os
from concurrent.futures import ThreadPoolExecutor, as_completed

In [ ]:
repo_root = os.path.dirname(os.path.dirname(os.path.dirname(os.path.dirname(os.getcwd()))))
config_dir = os.path.join(repo_root, "config", "data_ingest")

with open(os.path.join(config_dir, "catalog_config.json")) as f:
    catalog = json.load(f)["catalog"]

daily_batch_config_path = os.path.join(config_dir, "daily_batch_config.json")
with open(daily_batch_config_path) as f:
    batch_config = json.load(f)

triggers = batch_config["triggers"]

In [ ]:
spark.sql(f"""CREATE TABLE IF NOT EXISTS {catalog}.yfinance.nrt_statistics (
    ticker STRING,
    window_start STRING,
    window_end STRING,
    high DOUBLE,
    low DOUBLE,
    last_price DOUBLE,
    pct_change DOUBLE,
    vwap DOUBLE,
    volume DOUBLE
)
TBLPROPERTIES (
  'delta.schema.autoMerge.enabled' = 'true'
)
""")

In [ ]:
def run_worker(trigger):
    result = dbutils.notebook.run(
        "./worker",
        int(timeout) + 600,
        {
            "ticker": trigger["ticker"],
            "catalog": catalog,
            "timeout" : timeout
        },
    )
    return trigger["ticker"], result

results = {}
with ThreadPoolExecutor(max_workers=min(len(triggers), 8)) as executor:
    futures = [executor.submit(run_worker, trigger) for trigger in triggers]
    for future in as_completed(futures):
        ticker, result = future.result()
        results[ticker] = json.loads(result)

In [ ]:
failed = [ticker for ticker, result in results.items() if result["status"] != 1]
if failed:
    raise Exception(f"nrt statistics failed for: {failed}")